# Evaluation A1 — IID AMP Results

This notebook is the concise paper-facing view of finalized Evaluation A1. It loads canonical analysis artifacts produced from the frozen predictions; it does **not** implement metrics, bootstrap confidence intervals, sensitivity rules, or plots.

The reference labels are **SHERLOC silver-reference labels**, not ground truth. CPMR describes reference-contained prediction behavior and does not establish absolute factual correctness.

## 1. Setup and finalized-artifact gate

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, SVG, display


def locate_repo_root() -> Path:
    configured = os.environ.get("SHERLOC_REPO_ROOT")
    starts = [Path(configured).expanduser()] if configured else []
    starts.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in starts:
        if (candidate / "src/experiments/11_evaluate_amp.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate SHERLOC_Case_Analysis. Start Jupyter in the repository "
        "or set SHERLOC_REPO_ROOT."
    )


REPO_ROOT = locate_repo_root()
ANALYSIS_ROOT = REPO_ROOT / "outputs/analysis/evaluation_a"
FIGURE_ROOT = REPO_ROOT / "outputs/figures/evaluation_a"
METRICS_ROOT = REPO_ROOT / "outputs/metrics"


def load_csv(path: Path, required_columns=()) -> pd.DataFrame:
    """Load a finalized artifact without synthesizing missing rows."""
    if not path.is_file():
        display(Markdown(f"> **NOT YET AVAILABLE:** `{path.relative_to(REPO_ROOT)}`"))
        return pd.DataFrame(columns=list(required_columns))
    frame = pd.read_csv(path)
    missing = set(required_columns) - set(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")
    return frame


def load_json(path: Path) -> dict:
    if not path.is_file():
        display(Markdown(f"> **NOT YET AVAILABLE:** `{path.relative_to(REPO_ROOT)}`"))
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def show_table(frame: pd.DataFrame, *, empty_message="No finalized rows are available."):
    if frame.empty:
        display(Markdown(f"> **NOT YET AVAILABLE:** {empty_message}"))
    else:
        display(frame)


def show_figure(filename: str):
    """Display a finalized SVG; never recreate a figure in the notebook."""
    path = FIGURE_ROOT / filename
    if not path.is_file():
        display(Markdown(f"> **NOT YET AVAILABLE:** `{path.relative_to(REPO_ROOT)}`"))
        return
    display(SVG(filename=str(path)))


manifest = load_json(METRICS_ROOT / "amp_evaluation_manifest.json")
completion_gate = manifest.get("final_completion_gate", "NOT YET AVAILABLE")
display(Markdown(f"**Canonical Evaluation A completion gate:** `{completion_gate}`"))


In [ ]:
table_names = ['a1_main_comparison.csv', 'amp_family_level_metrics.csv', 'prediction_breadth_summary.csv', 'rare_label_sensitivity.csv', 'm3_vs_m4_summary.csv', 'm3_vs_m4_per_label_f1.csv']
figure_names = ['figure_1_a1_vs_a2_core_performance.svg', 'figure_2_cpmr_by_amp_family.svg']
artifact_inventory = pd.DataFrame([
    *[
        {"artifact": str((ANALYSIS_ROOT / name).relative_to(REPO_ROOT)),
          "kind": "paper-facing table", "available": (ANALYSIS_ROOT / name).is_file()}
        for name in table_names
    ],
    *[
        {"artifact": str((FIGURE_ROOT / name).relative_to(REPO_ROOT)),
          "kind": "finalized figure", "available": (FIGURE_ROOT / name).is_file()}
        for name in figure_names
    ],
])
display(artifact_inventory)
if not artifact_inventory["available"].all():
    display(Markdown("> **NOT YET AVAILABLE:** one or more finalized artifacts are missing."))


## 2. Canonical A1 main comparison

In [ ]:
a1_main = load_csv(ANALYSIS_ROOT / "a1_main_comparison.csv", ("method", "n"))
show_table(a1_main)


## 3. Family-level performance and prediction breadth

These are finalized descriptive summaries. Prediction breadth is not a primary performance metric.

In [ ]:
family_metrics = load_csv(
    ANALYSIS_ROOT / "amp_family_level_metrics.csv", ("evaluation", "method", "family")
)
prediction_breadth = load_csv(
    ANALYSIS_ROOT / "prediction_breadth_summary.csv", ("evaluation", "method")
)
show_table(family_metrics.loc[family_metrics["evaluation"].eq("A1")])
show_table(prediction_breadth.loc[prediction_breadth["evaluation"].eq("A1")])


## 4. Rare-label sensitivity analysis

This section is explicitly **descriptive sensitivity analysis**. It does not replace the frozen official Macro-F1.

In [ ]:
rare_sensitivity = load_csv(
    ANALYSIS_ROOT / "rare_label_sensitivity.csv", ("evaluation", "method")
)
show_table(rare_sensitivity.loc[rare_sensitivity["evaluation"].eq("A1")])


## 5. Descriptive M4 minus M3 comparison

In [ ]:
m3_m4 = load_csv(ANALYSIS_ROOT / "m3_vs_m4_summary.csv", ("evaluation",))
m3_m4_per_label = load_csv(
    ANALYSIS_ROOT / "m3_vs_m4_per_label_f1.csv", ("evaluation", "label_id")
)
show_table(m3_m4.loc[m3_m4["evaluation"].eq("A1")])
show_table(m3_m4_per_label.loc[m3_m4_per_label["evaluation"].eq("A1")])
display(Markdown("No statistical-significance claim is made for these descriptive differences."))


## 6. Core figures

In [ ]:
show_figure("figure_1_a1_vs_a2_core_performance.svg")
show_figure("figure_2_cpmr_by_amp_family.svg")


## 7. Interpretation boundary

Use the finalized tables as the numeric source for paper writing. The figures are presentation views of those same artifacts. Do not tune any frozen model, threshold, prompt, demo bank, split, or ontology from this notebook.